# Week 4: Enterprise REST API, Prescriptive Serving & Covariate Drift Diagnostics
## End-to-End MLOps Packaging, Real-Time Inference & Statistical Causal Health Auditing

Welcome to the **Week 4 Enterprise Serving & MLOps Platform** of the **EconoCausal** system.
In Weeks 1 through 3, we:
1. Formulated the Causal Directed Acyclic Graph (DAG) and synthesized dynamic pricing data ($0, $10, $20).
2. Trained an EconML Double Machine Learning (`LinearDML` + `LightGBM`) engine, passed a 3-part DoWhy adversarial refutation audit, and evaluated Qini/Uplift curves.
3. Formulated and solved the Multiple-Choice Knapsack Problem (MCKP) via SciPy HiGHS MILP, demonstrating a **+72.4% conversion efficiency lift** and **42.0% cost reduction** over blanket targeting.

In **Week 4**, we package these components into a production-grade microservice architecture:
- **FastAPI REST Serving:** Real-time single-customer checkout scoring and high-throughput batch inference.
- **Dynamic Prescriptive Optimization:** Automated Knapsack discount allocation under arbitrary budget constraints.
- **Covariate Shift & Drift Diagnostics:** Two-Sample Kolmogorov-Smirnov (KS) tests and Population Stability Index (PSI) to guard against causal identification collapse.


In [ ]:
import os
import json
import time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Set aesthetics
sns.set_theme(style="whitegrid")
plt.rcParams['font.family'] = 'DejaVu Sans'
plt.rcParams['figure.figsize'] = (10, 5)

from api import (
    CustomerPayload,
    BatchPredictionRequest,
    OptimizationRequest,
    read_root,
    health_check,
    predict_individual_customer,
    predict_batch_customers,
    optimize_campaign_allocation
)
from drift_detector import (
    calculate_psi,
    calculate_ks_test,
    load_baseline_dataset,
    generate_production_cohort,
    evaluate_cohort_drift,
    export_drift_report
)

print("EconoCausal Production API & Drift Modules loaded successfully!")


### 1. System Healthcheck & Solver Readiness (`GET /health`)
We verify microservice health, loaded model parameters, and active optimization solvers (`scipy.optimize.milp` and `solve_prescriptive_greedy`).


In [ ]:
health = health_check()
print(f"Service Status: {health.status.upper()}")
print(f"Uptime:         {health.uptime_seconds:.2f} seconds")
print(f"Active Solvers: {', '.join(health.active_solvers)}")
print(f"API Version:    {health.version}")


### 2. Real-Time Single Customer Personalization (`POST /predict/individual`)
Simulates how an e-commerce checkout page or mobile app queries EconoCausal to decide the optimal promotional tier ($0, $10, or $20) for an individual shopper based on causal price elasticity.


In [ ]:
# Test Case A: Young customer (<30) - High Elasticity Persuadable
p_young = CustomerPayload(client_id="client_young_01", age=23.0, has_redeemed=1, issue_redeem_delay=6.0)
res_young = predict_individual_customer(p_young)

# Test Case B: Senior customer (>50) - Moderate Elasticity
p_senior = CustomerPayload(client_id="client_senior_02", age=68.0, has_redeemed=0, issue_redeem_delay=2.0)
res_senior = predict_individual_customer(p_senior)

print("--- SINGLE CUSTOMER INFERENCE RESULTS ---")
print(f"Young Customer ({p_young.age} yrs):")
print(f"  ITE $10: {res_young.ite_10:.4f} | ITE $20: {res_young.ite_20:.4f}")
print(f"  Prescribed Tier: ${res_young.recommended_discount} (Expected Lift: {res_young.expected_lift_pct})")
print(f"  Segment:         {res_young.price_elasticity_segment}
")

print(f"Senior Customer ({p_senior.age} yrs):")
print(f"  ITE $10: {res_senior.ite_10:.4f} | ITE $20: {res_senior.ite_20:.4f}")
print(f"  Prescribed Tier: ${res_senior.recommended_discount} (Expected Lift: {res_senior.expected_lift_pct})")
print(f"  Segment:         {res_senior.price_elasticity_segment}")


### 3. High-Throughput Batch Cohort Inference (`POST /predict/batch`)
For batch marketing drops (e.g. weekly CRM push to 10,000 customers), EconoCausal uses vectorized matrix operations to score thousands of shoppers in milliseconds.


In [ ]:
# Generate cohort of 500 diverse customers
np.random.seed(42)
cohort_customers = [
    CustomerPayload(
        client_id=f"cohort_cli_{i:04d}",
        age=float(np.random.normal(44, 15)),
        has_redeemed=int(np.random.binomial(1, 0.4)),
        issue_redeem_delay=float(np.random.exponential(10))
    )
    for i in range(500)
]

t0 = time.time()
batch_req = BatchPredictionRequest(customers=cohort_customers)
batch_res = predict_batch_customers(batch_req)
batch_time = (time.time() - t0) * 1000

print(f"Processed {batch_res.total_processed:,} customers in {batch_time:.2f} ms!")
print(f"Average Treatment Effect (ATE) - $10 Discount: +{batch_res.mean_ate_10_pct:.2f}%")
print(f"Average Treatment Effect (ATE) - $20 Discount: +{batch_res.mean_ate_20_pct:.2f}%")

# Visualize predicted ITE distributions
df_pred_plot = pd.DataFrame([
    {'ite_10': p.ite_10, 'ite_20': p.ite_20, 'discount': p.recommended_discount}
    for p in batch_res.predictions
])

plt.figure(figsize=(10, 4))
sns.histplot(df_pred_plot['ite_10'], color='#3498db', label='ITE $10 Lift', kde=True, stat="density", bins=25)
sns.histplot(df_pred_plot['ite_20'], color='#2ecc71', label='ITE $20 Lift', kde=True, stat="density", bins=25)
plt.title("Distribution of Individual Treatment Effects (ITE) Across 500 Customers", fontsize=12, fontweight='bold')
plt.xlabel("Incremental Purchase Lift", fontsize=11)
plt.ylabel("Density", fontsize=11)
plt.legend()
plt.tight_layout()
plt.show()


### 4. Dynamic Prescriptive Knapsack Optimization (`POST /optimize`)
We invoke the prescriptive solver endpoint under a $5,000 campaign budget to allocate discount tiers and benchmark efficiency against alternative strategies.


In [ ]:
opt_req = OptimizationRequest(
    budget=5000.0,
    cost_10=10.0,
    cost_20=20.0,
    solver="greedy"
)

t0 = time.time()
opt_res = optimize_campaign_allocation(opt_req)
opt_time = (time.time() - t0) * 1000

print("--- PRESCRIPTIVE ALLOCATION RESULTS ($5,000 BUDGET) ---")
print(f"Total Budget Allocated:        ${opt_res.budget_allocated:,.2f} ({opt_res.budget_utilization_pct}%)")
print(f"Incremental Buyers Gained:     {opt_res.incremental_buyers_gained:.1f} buyers")
print(f"Cost per Incremental Buyer:    ${opt_res.cost_per_incremental_buyer:.2f}")
print(f"Efficiency Lift vs Blanket:    +{opt_res.efficiency_lift_vs_blanket_pct:.1f}%")
print(f"Execution Latency:             {opt_res.execution_time_ms:.2f} ms
")

print("Policy Benchmark Matrix:")
bench_df = pd.DataFrame([b.model_dump() for b in opt_res.policy_benchmarks])
bench_df


### 5. Statistical Covariate Shift & Data Drift Monitoring (`POST /drift/evaluate`)
Why Data Drift is Critical in Causal AI:
If the distribution of confounders ($W$) or modifiers ($X$) changes, the conditional exchangeability assumption ($Y(t) \perp T \mid W$) can fail.
We compute the **Two-Sample Kolmogorov-Smirnov (KS)** test and the **Population Stability Index (PSI)** to compare live traffic against the training baseline.


In [ ]:
base_df = load_baseline_dataset(sample_size=5000)
prod_stable = generate_production_cohort(scenario="stable", n=2500)
prod_severe = generate_production_cohort(scenario="severe", n=2500)

report_stable = evaluate_cohort_drift(base_df, prod_stable)
report_severe = evaluate_cohort_drift(base_df, prod_severe)

print(f"Normal Cohort Verdict:  {report_stable['badge']} (Mean PSI: {report_stable['overall_psi']:.4f})")
print(f"Shifted Cohort Verdict: {report_severe['badge']} (Mean PSI: {report_severe['overall_psi']:.4f})")
print(f"Critical Alert:
{report_severe['recommendation']}")

# Side-by-side distribution plots
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

sns.kdeplot(base_df['age'], ax=axes[0], label='Baseline Training', color='#2980b9', fill=True, alpha=0.3)
sns.kdeplot(prod_stable['age'], ax=axes[0], label='Stable Cohort', color='#27ae60', linestyle='--')
axes[0].set_title(f"Stable Traffic (Age PSI: {report_stable['feature_metrics']['age']['psi_score']:.3f})", fontweight='bold')
axes[0].set_xlabel("Age (Years)")
axes[0].legend()

sns.kdeplot(base_df['age'], ax=axes[1], label='Baseline Training', color='#2980b9', fill=True, alpha=0.3)
sns.kdeplot(prod_severe['age'], ax=axes[1], label='Severe Shift (Young Influx)', color='#c0392b', fill=True, alpha=0.3)
axes[1].set_title(f"Severe Drift Detected (Age PSI: {report_severe['feature_metrics']['age']['psi_score']:.3f})", fontweight='bold')
axes[1].set_xlabel("Age (Years)")
axes[1].legend()

plt.tight_layout()
plt.show()


### 6. Audit Telemetry & Governance Report
We export the final telemetry audit report to `data/drift_report.json` for ingestion into enterprise observability platforms (e.g. Datadog, Prometheus).


In [ ]:
export_drift_report(report_stable, output_path="data/drift_report.json")
print("Drift audit report serialized to data/drift_report.json.")

with open("data/drift_report.json", "r") as f:
    saved_rep = json.load(f)
    
print("\nAudit Report Header:")
print(f"  Timestamp:    {saved_rep['timestamp']}")
print(f"  Status:       {saved_rep['overall_status']}")
print(f"  Mean PSI:     {saved_rep['overall_psi']}")
print(f"  Features:     {list(saved_rep['feature_metrics'].keys())}")
